In [ ]:
!pip install PyMC

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_regression, SelectKBest, SelectFromModel
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, ARDRegression, LassoLarsCV
import pymc as pm
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor 
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, r2_score

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


In [2]:
df = pd.read_csv('imputed_iqr.csv')

In [3]:
len(df)

1384

In [4]:
df.head(20)

,Dev Time (Days),Start Date,total_swag,impacted_area_(none),impacted_area_3rd Party,impacted_area_AMS,impacted_area_ASTRO Infrastructure,impacted_area_Accessories,impacted_area_Architecture,impacted_area_Audio,...,Engineering,One list,PDM,Parity/Evolution,Quality,Quality-ENG,Roadmap/PCN,SLT,Security,subtask_count
0,188.0,20200505.0,361.447,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,188.0,20200505.0,383.370,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,188.0,20200720.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0
3,188.0,20201215.0,285.577,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,189.0,20220118.0,14.400,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
5,189.0,20220913.0,1810.058,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,190.0,20200130.0,373.737,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,190.0,20220516.0,114.170,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,190.0,20230302.0,30.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
9,190.0,20230418.0,675.049,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Method 1: Mutual Information (Filter method)

In [5]:
X = df.drop(columns=['Dev Time (Days)'], axis=1)
y = df['Dev Time (Days)']



In [6]:
mi_scores = mutual_info_regression(X, y, random_state=42)

mi_series = pd.Series(mi_scores, index=X.columns, name="MI Scores")
mi_series = mi_series.sort_values(ascending=False)

print("\nMutual Information Regression Scores:")
print(mi_series)


Mutual Information Regression Scores:
Start Date                       1.727862
total_swag                       0.762194
fix_version_(no version)         0.083025
impacted_area_(none)             0.080992
technology_(none)                0.074411
                                   ...   
fix_version_ASTRO_SR2025.1MR     0.000000
fix_version_ASTRO_SR2024.1CSR    0.000000
fix_version_ASTRO_SR2024.1       0.000000
fix_version_ASTRO_SR2024.2CSR    0.000000
fix_version_RadioNext 2.0        0.000000
Name: MI Scores, Length: 176, dtype: float64


In [7]:
len(mi_series)

176

We select features with mi score > 0.01

In [8]:
good_features = mi_series[mi_series > 0.01].index


X_mutual_info = X[good_features]

print(X_mutual_info.columns)

listm = X_mutual_info.columns
len(listm)

Index(['Start Date', 'total_swag', 'fix_version_(no version)',
       'impacted_area_(none)', 'technology_(none)', 'technology_APX',
       'affects_version_(no version)', 'subtask_count', 'technology_Aloha',
       'fix_version_ASTRO_SR7.18.5', 'fix_version_ASTRO_SR2022.2',
       'fix_version_ASTRO_SR2019.2', 'fix_version_ASTRO_SR7.18.8',
       'affects_version_ASTRO_SR2025.1', 'Quality', 'impacted_area_UI / UX',
       'technology_APX Next', 'One list', 'affects_version_No_SW_Release',
       'affects_version_ASTRO_SR2026.2', 'affects_version_ASTRO_SR2021.3',
       'Cyber RR', 'impacted_area_Manufacturing', 'impacted_area_GCD',
       'Security', 'fix_version_ASTRO_SR2023.2', 'impacted_area_Tools',
       'impacted_area_DSP / Audio', 'impacted_area_Encryption',
       'fix_version_No_SW_Release', 'Quality-ENG',
       'fix_version_ASTRO_SR2019.4', 'impacted_area_CPE-EE',
       'impacted_area_CC CAD', 'fix_version_ASTRO_SR2022.1MR1'],
      dtype='object')


35

# Method 2: Lasso Regression (Embedded Method)

First, scale the data

In [9]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X, y)

In [10]:
lasso_cv = LassoCV(cv=5, random_state=42)

In [11]:
selector = SelectFromModel(estimator=lasso_cv, threshold=1e-5)

In [12]:
selector.fit(X_scaled, y)

SelectFromModel(estimator=LassoCV(cv=5, random_state=42), threshold=1e-05)

In [13]:
selected_cols = X.columns[selector.get_support()]
X_lasso = X[selected_cols]
print(X_lasso.columns)

list2 = X_lasso.columns
len(list2)

Index(['total_swag', 'impacted_area_(none)', 'impacted_area_3rd Party',
       'impacted_area_Architecture', 'impacted_area_Audio',
       'impacted_area_CC Aware', 'impacted_area_CC Messaging',
       'impacted_area_CM', 'impacted_area_CMSO', 'impacted_area_CPE',
       'impacted_area_CPE-EE', 'impacted_area_CPS/RM',
       'impacted_area_Connectivity', 'impacted_area_DCE',
       'impacted_area_EE - RF', 'impacted_area_Encryption',
       'impacted_area_Energy', 'impacted_area_Factory',
       'impacted_area_Harmony Portal', 'impacted_area_IDM', 'impacted_area_IT',
       'impacted_area_IoT Hub', 'impacted_area_Manufacturing',
       'impacted_area_Mechanical', 'impacted_area_RC Insights',
       'impacted_area_RC Multi-Tenant', 'impacted_area_RC Platform',
       'impacted_area_RLS', 'impacted_area_Radio Central', 'impacted_area_SPG',
       'impacted_area_SW AP MIL', 'impacted_area_SWBB Android Core / Common',
       'impacted_area_SWBB Connectivity', 'impacted_area_Signalling',
  

63

# Method 3: Automatic Relevancy Determination (Embedded Method - Bayesian)

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit ARD Regression model
ard = ARDRegression(compute_score=True)
ard.fit(X_train, y_train)

# Extract feature weights and select relevent features
feature_weights = pd.Series(ard.coef_, index=X.columns)
relevant_features = feature_weights[np.abs(feature_weights) > 0.01]

print(f"Total initial features: {len(X.columns)}")
print(f"Features kept by ARD: {len(relevant_features)}")
print(f"Current n/p ratio: {len(X_train) / len(relevant_features):.2f}")


Total initial features: 176
Features kept by ARD: 56
Current n/p ratio: 19.77


# Method 4: Submodular Optimization (Wrapper)

In [15]:
import numpy as np
import pandas as pd
from sklearn.metrics import mutual_info_score
from sklearn.feature_selection import mutual_info_regression

import numpy as np
import pandas as pd
from sklearn.metrics import mutual_info_score
from sklearn.feature_selection import mutual_info_regression

def submodular_greedy_selection(X_data, y_target, k):
    all_features = list(X_data.columns)
    selected_indices = []
    remaining_indices = list(range(len(all_features)))
    
    k = min(k, len(all_features))
    print(f"--- Starting Submodular Selection (Budget k={k}) ---")

    for i in range(k):
        best_gain = -np.inf
        best_feature_idx = -1

        for idx in remaining_indices:
            # RELEVANCE: Always use regression MI for continuous target y
            current_feat_vals = X_data.iloc[:, [idx]].values
            relevance = mutual_info_regression(current_feat_vals, y_target, random_state=42)[0]
            
            # REDUNDANCY: Logic branch based on feature type
            redundancy = 0
            if selected_indices:
                red_scores = []
                for s in selected_indices:
                    feat_a = X_data.iloc[:, idx]
                    feat_b = X_data.iloc[:, s]
                    
                    # If BOTH are binary/categorical, use mutual_info_score (Discrete)
                    if len(np.unique(feat_a)) < 10 and len(np.unique(feat_b)) < 10:
                        score = mutual_info_score(feat_a, feat_b)
                    else:
                        # If either is continuous, use regression-based MI (Continuous)
                        # We reshape to (n, 1) for the regression function
                        score = mutual_info_regression(feat_a.values.reshape(-1, 1), feat_b, random_state=42)[0]
                    
                    red_scores.append(score)
                redundancy = np.mean(red_scores)
            
          
            gain = relevance - redundancy

            if gain > best_gain:
                best_gain = gain
                best_feature_idx = idx

        if best_feature_idx != -1:
            # EARLY STOPPING: If gain is negative, adding the feature makes the set worse
            if best_gain <= 0:
                print(f"--- Stopping early at Step {i+1}: Gain became non-positive ---")
                break
                
            selected_indices.append(best_feature_idx)
            remaining_indices.remove(best_feature_idx)
            print(f"Step {i+1}: Added '{all_features[best_feature_idx]}' | Gain: {best_gain:.4f}")
        else:
            break

    final_selection = [all_features[i] for i in selected_indices]
    print(f"\nFinal Feature Set ({len(final_selection)}): {final_selection}")
    return final_selection


final_features = submodular_greedy_selection(X, y, k=18) # Set k=18 because gain turned negative at k=19


--- Starting Submodular Selection (Budget k=18) ---
Step 1: Added 'Start Date' | Gain: 1.7242
Step 2: Added 'total_swag' | Gain: 0.0290
Step 3: Added 'Quality-ENG' | Gain: 0.0047
Step 4: Added 'fix_version_ASTRO_SR7.18.5' | Gain: 0.0021
Step 5: Added 'fix_version_ASTRO_SR2019.2' | Gain: 0.0059
Step 6: Added 'fix_version_(no version)' | Gain: 0.0103
Step 7: Added 'impacted_area_(none)' | Gain: 0.0135
Step 8: Added 'fix_version_ASTRO_SR7.18.8' | Gain: 0.0018
Step 9: Added 'impacted_area_CPE-EE' | Gain: 0.0000
Step 10: Added 'affects_version_(no version)' | Gain: 0.0003
Step 11: Added 'technology_APX' | Gain: 0.0014
--- Stopping early at Step 12: Gain became non-positive ---

Final Feature Set (11): ['Start Date', 'total_swag', 'Quality-ENG', 'fix_version_ASTRO_SR7.18.5', 'fix_version_ASTRO_SR2019.2', 'fix_version_(no version)', 'impacted_area_(none)', 'fix_version_ASTRO_SR7.18.8', 'impacted_area_CPE-EE', 'affects_version_(no version)', 'technology_APX']


# Deep Knock-Off

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np

class KnockoffVAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(KnockoffVAE, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim * 2) # Mean and variance
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim),
            nn.Sigmoid() # Keeps values between 0-1 (good for scaled data)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = torch.chunk(h, 2, dim=1)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

def generate_deep_knockoffs(X_df, epochs=100, latent_dim=10):
    X_min = X_df.min()
    X_max = X_df.max()
    # Handle columns with zero variance
    X_range = (X_max - X_min).replace(0, 1)
    X_scaled = (X_df - X_min) / X_range
    
    X_tensor = torch.FloatTensor(X_scaled.values)
    input_dim = X_tensor.shape[1]
    
    #Setup model
    model = KnockoffVAE(input_dim, latent_dim)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    #Training Loop
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        recon_x, mu, logvar = model(X_tensor)
        
        #Loss = Reconstruction Error + KL Divergence
        recon_loss = nn.functional.mse_loss(recon_x, X_tensor, reduction='sum')
        kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        loss = recon_loss + kld_loss
        
        loss.backward()
        optimizer.step()

    #Generate Knockoffs
    model.eval()
    with torch.no_grad():
        # Re-encode and sample to create the knockoff twins
        X_tilde_scaled, _, _ = model(X_tensor)
        X_tilde_np = X_tilde_scaled.numpy()
        
    #Reverse Scaling
    X_tilde = (X_tilde_np * X_range.values) + X_min.values
    
    #Clean up Binary columns (Clip to 0 or 1 for one-hot encoded fields)
    for i, col in enumerate(X_df.columns):
        if len(X_df[col].unique()) <= 2: 
            X_tilde[:, i] = np.where(X_tilde[:, i] > 0.5, 1, 0)

    return pd.DataFrame(X_tilde, columns=[f"knockoff_{c}" for c in X_df.columns])

X_knockoff = generate_deep_knockoffs(X)

In [17]:
X_combined = pd.concat([X,X_knockoff], axis=1)

In [19]:
mi_scores = mutual_info_regression(X_combined, y, random_state=42)

n_features = X.shape[1]
Z = mi_scores[:n_features] # real mi scores
Z_tilde = mi_scores[n_features:] # Knockoff mi scores

W = Z - Z_tilde # difference in information

final_mask = (Z > 0.01) & (W > 0)
valid_indices = np.where(final_mask)[0]

mi_good_features_knockoff = X.columns[valid_indices]
X_mutual_info_final = X[mi_good_features_knockoff]

print(f"Original MI selection count: {len(mi_series[mi_series > 0.01])}")
print(f"Knockoff-validated MI count: {len(mi_good_features_knockoff)}")
print("\nValidated Features:")
print(mi_good_features_knockoff.tolist())

Original MI selection count: 35
Knockoff-validated MI count: 35

Validated Features:
['Start Date', 'total_swag', 'impacted_area_(none)', 'impacted_area_ASTRO Infrastructure', 'impacted_area_Energy', 'impacted_area_Mechanical', 'impacted_area_SWBB Connectivity', 'impacted_area_SWBB Low Level', 'technology_(none)', 'technology_APX', 'technology_Accessories', 'technology_Aloha', 'technology_Energy', 'affects_version_(no version)', 'affects_version_ASTRO_SR2020.3', 'affects_version_ASTRO_SR2020.4', 'affects_version_ASTRO_SR2021.3', 'affects_version_ASTRO_SR2023.1MR1', 'affects_version_ASTRO_SR2023.2', 'affects_version_ASTRO_SR2025.1', 'affects_version_ASTRO_SR2026.1', 'affects_version_No_SW_Release', 'fix_version_(no version)', 'fix_version_ASTRO_SR2019.2', 'fix_version_ASTRO_SR2021.4', 'fix_version_ASTRO_SR2022.2', 'fix_version_ASTRO_SR2024.1', 'fix_version_ASTRO_SR2024.1MR', 'fix_version_ASTRO_SR2024.2MR', 'fix_version_ASTRO_SR7.18.5', 'fix_version_ASTRO_SR7.18.8', 'PDM', 'Quality-ENG',

In [20]:
selected_features_combined = submodular_greedy_selection(X_combined, y, k=36)


selection_results = {feat: i for i, feat in enumerate(selected_features_combined)}

real_winners = []
knockoff_interlopers = []
final_selection = []

original_features = X.columns.tolist()

print("\n--- Submodular Knockoff Comparison ---")

for feat in original_features:
    knock_feat = f"knockoff_{feat}"

    rank_real = selection_results.get(feat, float('inf'))
    rank_knockoff = selection_results.get(knock_feat, float('inf'))
    

    if rank_real < rank_knockoff:
        real_winners.append(feat)
        final_selection.append(feat)
    elif rank_knockoff < rank_real:
        knockoff_interlopers.append(feat)

print(f"Total original features tested: {len(original_features)}")
print(f"Features that beat their knockoff: {len(real_winners)}")
print(f"Features beaten by their knockoff (Noise): {len(knockoff_interlopers)}")

print("\nFinal Validated Feature Set:")
print(final_selection)

if knockoff_interlopers:
    print("\nRejected (Noise) Features:")
    print(knockoff_interlopers)

--- Starting Submodular Selection (Budget k=36) ---
Step 1: Added 'Start Date' | Gain: 1.7242
Step 2: Added 'total_swag' | Gain: 0.0290
Step 3: Added 'Quality-ENG' | Gain: 0.0047
Step 4: Added 'knockoff_total_swag' | Gain: 0.0076
Step 5: Added 'fix_version_ASTRO_SR7.18.5' | Gain: 0.0061
Step 6: Added 'fix_version_(no version)' | Gain: 0.0136
Step 7: Added 'impacted_area_(none)' | Gain: 0.0176
Step 8: Added 'fix_version_ASTRO_SR2019.2' | Gain: 0.0088
Step 9: Added 'fix_version_ASTRO_SR7.18.8' | Gain: 0.0039
Step 10: Added 'affects_version_(no version)' | Gain: 0.0003
Step 11: Added 'technology_APX' | Gain: 0.0014
Step 12: Added 'impacted_area_CPE-EE' | Gain: 0.0002
--- Stopping early at Step 13: Gain became non-positive ---

Final Feature Set (12): ['Start Date', 'total_swag', 'Quality-ENG', 'knockoff_total_swag', 'fix_version_ASTRO_SR7.18.5', 'fix_version_(no version)', 'impacted_area_(none)', 'fix_version_ASTRO_SR2019.2', 'fix_version_ASTRO_SR7.18.8', 'affects_version_(no version)', '

In [84]:
X_combined_scaled = scaler.fit_transform(X_combined)

lasso_cv = LassoCV(
    n_alphas=200,
    cv=5, 
    random_state=42, 
    max_iter=50000,    
    tol=0.01         
)

lasso_cv.fit(X_combined_scaled,y)

n_features = X.shape[1]
abs_coefs = np.abs(lasso_cv.coef_)

z_real = abs_coefs[:n_features]
z_knockoff = abs_coefs[:n_features]

w_stats = z_real - z_knockoff
real_winners = X.columns[w_stats > 0].tolist()

beaten_by_noise = X.columns[z_knockoff > z_real].tolist()

print("--- Lasso Knockoff Validation ---")
print(f"Features originally in X: {n_features}")
print(f"Features that beat their twin: {len(real_winners)}")
print(f"Features beaten by noise: {len(beaten_by_noise)}")

print("\nFinal Validated Lasso Set:")
print(real_winners)

if beaten_by_noise:
    print("\nRejected Features (Interlopers):")
    print(beaten_by_noise)

--- Lasso Knockoff Validation ---
Features originally in X: 176
Features that beat their twin: 0
Features beaten by noise: 0

Final Validated Lasso Set:
[]


LASSO is not a good model for feature selection for this dataset

In [21]:
from sklearn.linear_model import ARDRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_comb_scaled = scaler.fit_transform(X_combined)

# Fit ARD
print("--- Starting ARD Knockoff Tournament ---")
ard_model = ARDRegression(compute_score=True)
ard_model.fit(X_comb_scaled, y)

n_features = X.shape[1]
weights = np.abs(ard_model.coef_)

z_real = weights[:n_features]           # Importance of real features
z_knockoff = weights[n_features:]      # Importance of synthetic twins

w_stats = z_real - z_knockoff
valid_mask = (w_stats > 0) & (z_real > 0.01)
real_winners_ard = X.columns[valid_mask].tolist()


print(f"Features originally in X: {n_features}")
print(f"ARD Survivors (Beat their knockoff): {len(real_winners_ard)}")

if len(real_winners_ard) > 0:
    print("\nValidated ARD Features:")
    print(real_winners_ard)
else:
    print("\nNo features survived the ARD competition.")

--- Starting ARD Knockoff Tournament ---
Features originally in X: 176
ARD Survivors (Beat their knockoff): 58

Validated ARD Features:
['total_swag', 'impacted_area_(none)', 'impacted_area_3rd Party', 'impacted_area_Architecture', 'impacted_area_Audio', 'impacted_area_CC Messaging', 'impacted_area_CM', 'impacted_area_CMSO', 'impacted_area_CPE', 'impacted_area_CPE-EE', 'impacted_area_Connectivity', 'impacted_area_DCE', 'impacted_area_EE - RF', 'impacted_area_Energy', 'impacted_area_Harmony Portal', 'impacted_area_IDM', 'impacted_area_IT', 'impacted_area_IoT Hub', 'impacted_area_Manufacturing', 'impacted_area_Mechanical', 'impacted_area_RC Insights', 'impacted_area_RC Multi-Tenant', 'impacted_area_RC Platform', 'impacted_area_RLS', 'impacted_area_Radio Central', 'impacted_area_SPG', 'impacted_area_SW AP MIL', 'impacted_area_SWBB Android Core / Common', 'impacted_area_SWBB Connectivity', 'impacted_area_Signalling', 'technology_APX NEXT XN', 'technology_APX Next', 'technology_Mahalo', 'af

In [22]:
mutual = pd.concat([y,X[mi_good_features_knockoff]],axis=1)
ARD = pd.concat([y,X[real_winners_ard]],axis=1)
submodular = pd.concat([y,X[final_selection]],axis=1)

In [23]:
mutual.to_csv('iqr_mutual_info.csv',index=False)
ARD.to_csv('iqr_ard.csv',index=False)
submodular.to_csv('iqr_submod.csv',index=False)